# Tổng quan dữ liệu huấn luyện ADNI

Đặt notebook trong validate, chạy Run All từ đầu. Dữ liệu được đọc từ system/data/train. Mỗi bảng thể hiện một đơn vị cụ thể: hàng CSV, lượt khám, RID hoặc mẫu đủ điều kiện huấn luyện.

## 1. Xác định thư mục hệ thống

Notebook tìm system từ thư mục làm việc của kernel. Hỗ trợ validate nằm cạnh system hoặc bên trong system. Nếu mở Jupyter ở vị trí khác, điền đường dẫn tuyệt đối tới system vào SYSTEM_DIR.

In [ ]:
from pathlib import Path
import json
import sys
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

SYSTEM_DIR = None  # Ví dụ Path(r"D:\ALZHEIMER\Alzheimer\system")
cwd = Path.cwd()
candidates = [Path(SYSTEM_DIR)] if SYSTEM_DIR else [p for base in (cwd, *cwd.parents) for p in (base / "system", base)]
ROOT = next((p.resolve() for p in candidates if (p / "pipeline.py").is_file() and (p / "data/train").is_dir()), None)
if ROOT is None:
    raise FileNotFoundError("Hãy đặt SYSTEM_DIR tới thư mục system chứa pipeline.py và data/train.")
TRAIN = ROOT / "data/train"
sys.path.insert(0, str(ROOT))
from pipeline import MODELS, cohort
from registry import TABLES
print("Thư mục train:", TRAIN)

## 2. Kiểm kê CSV huấn luyện

Đếm hàng, cột và RID riêng cho từng bảng. DATADIC là từ điển biến. Tổng số người được tính bằng hợp các RID từ năm bảng đầu vào.

In [ ]:
files = {}
for name in [*TABLES, "DATADIC"]:
    matches = list(TRAIN.rglob(name + ".csv"))
    if len(matches) == 1:
        files[name] = matches[0]
    elif matches or name in TABLES:
        raise ValueError(f"Cần đúng một {name}.csv, tìm thấy {len(matches)}.")
inventory, nulls, ids = [], [], set()
for name, path in files.items():
    df = pd.read_csv(path, encoding="latin1", low_memory=False)
    people = set(pd.to_numeric(df.RID, errors="raise").dropna()) if name in TABLES else None
    if people is not None:
        ids.update(people)
    inventory.append([name,len(df),len(df.columns),len(people) if people is not None else None])
    for col in df:
        n = int(df[col].isna().sum())
        nulls.append([name,col,n,len(df),100*n/len(df) if len(df) else None])
display(pd.DataFrame(inventory, columns=["Bảng","Hàng","Cột","RID duy nhất"]))

## 3. Giá trị thiếu theo từng cột

Đếm null theo quy tắc mặc định của pandas khi đọc CSV. Mẫu số là số hàng trong chính bảng đó. Mã số quy ước như −4 vẫn được tính là giá trị có mặt. Bảng mặc định hiển thị 25 cột có tỷ lệ thiếu cao nhất. Chọn tên bảng để xem toàn bộ cột của bảng đó.

In [ ]:
missing_table = pd.DataFrame(nulls, columns=["Bảng","Cột","Null","Số hàng","Thiếu (%)"])
BANG = None  # Ví dụ "UCSFFSX7", "MMSE" hoặc "DATADIC"
ranked = missing_table.sort_values(["Thiếu (%)","Null"], ascending=False)
with pd.option_context("display.max_rows", None):
    display(ranked.head(25) if BANG is None else ranked[ranked["Bảng"].eq(BANG)])

## 4. Số người và lượt khám

Một lượt khám là một cặp RID và ngày hợp lệ trong DXSUM. Khoảng cách được tính giữa hai lượt liên tiếp của cùng người, rồi lấy trung vị trên các khoảng cách đó.

In [ ]:
dx = pd.read_csv(files["DXSUM"], usecols=["RID","EXAMDATE"])
dx["day"] = pd.to_datetime(dx.EXAMDATE, errors="coerce")
keys = dx.dropna(subset=["RID","day"]).drop_duplicates(["RID","day"]).sort_values(["RID","day"])
gaps = keys.groupby("RID").day.diff().dt.days
print("RID trong toàn bộ train:", len(ids))
print("Lượt khám DXSUM:", len(keys))
print("RID có ngày khám hợp lệ:", keys.RID.nunique())
print("Trung vị khoảng cách giữa hai lượt (ngày):", gaps.median())

## 5. Từ dữ liệu thô đến mốc dự báo

Gọi chính cohort() của hệ thống. Mỗi RID đủ điều kiện có một t0 tại lần MCI thứ hai ngoài sàng lọc. Bảng đếm số người có thời gian theo dõi dương, thời gian theo dõi trung vị và số người có chẩn đoán mâu thuẫn cùng ngày.

In [ ]:
frame, issues = cohort(TRAIN)
followed = frame.loc[frame.days.gt(0)].copy()
display(pd.Series({"RID nguồn":len(ids), "RID có t0":len(frame), "Theo dõi > 0 ngày":len(followed),
                   "Trung vị theo dõi (ngày)":followed.days.median(), "RID mâu thuẫn":issues.RID.nunique()},name="Train"))

## 6. Mẫu sử dụng cho từng mô hình M0–M3

Áp dụng đúng bộ lọc train.py: days > 0 rồi loại hàng thiếu bất kỳ biến nào của mô hình. Biến cố là lần đầu ghi nhận chẩn đoán mã 3 sau t0. Kiểm duyệt là chưa ghi nhận biến cố đến lần theo dõi cuối. Cột log và trạng thái fit lấy từ runs/models.json của lần huấn luyện đã lưu.

In [ ]:
log_path = ROOT / "runs/models.json"
log = json.loads(log_path.read_text(encoding="utf8")) if log_path.exists() else {}
rows = []
for model, cols in MODELS.items():
    usable = followed.dropna(subset=cols)
    events = int(usable.event.sum())
    saved = log.get(model,{})
    match = (len(usable),events)==(saved.get("n"),saved.get("events")) if saved else None
    rows.append([model,len(usable),len(followed)-len(usable),events,len(usable)-events,
                 saved.get("n"),saved.get("model") is not None if saved else None,match])
counts = pd.DataFrame(rows, columns=["Mô hình","Đủ điều kiện","Thiếu biến","Biến cố","Kiểm duyệt","N trong log","Đã fit","Khớp N và biến cố"])
display(pd.Series({m:", ".join(c) for m,c in MODELS.items()},name="Biến đầu vào").to_frame())
display(counts)
counts.set_index("Mô hình")["Đủ điều kiện"].plot.bar(rot=0,ylabel="Số người",title="Số người đủ dữ liệu theo mô hình")
plt.tight_layout()
plt.show()

## 7. Mức độ thiếu và phân bố của từng biến đầu vào

Mẫu số là số người trong train có theo dõi sau t0. Q1, trung vị và Q3 được tính trên các giá trị có mặt. Một người có thể thiếu nhiều biến nên các số thiếu không cộng dồn thành số người bị loại.

In [ ]:
x = followed[MODELS["M3"]]
stats = pd.DataFrame({"Số có":x.count(),"Số thiếu":x.isna().sum(),"Thiếu (%)":100*x.isna().mean(),
                      "Q1":x.quantile(.25),"Trung vị":x.median(),"Q3":x.quantile(.75)})
display(stats.rename_axis("Biến").sort_values("Số thiếu",ascending=False).round(2))